# Time Operators

Streaming data is infinite. Kafi Streams is an in-memory stream processor. Memory is never infinite.

So of course you need *time operators* that effectively clean up memory so that your Kafi Streams processing pipeline has constant, not ever-growing memory usage.

Freeing memory of timed out data is implemented as [expiry](#expiry) in Kafi Streams.

The time operators also allow you to implement the [*time windows*](#windows) as e.g. in Kafka Streams, such as [tumbling](#tumbling), [hopping](#hopping), [cumulative](#cumulative), [sliding](#sliding) and [session](#session) windows. And moreover, Kafi Streams enables you to build arbitrary [new types of time windows](#custom) as well.

## Overview

[Preparation](#prep)

* [Expiry](#expiry)
  * [expire()](#expire-operator)
* [Time windows in general](#windows)
  * [Time Windows = expire + group + aggregate](#expire_group_aggregate)
  * [Watermarks](#watermarks)
* [Tumbling windows](#tumbling)
  * [expire_tumbling()](#expire_tumbling-operator)
  * [group_by_agg_tumbling()](#group_by_agg_tumbling-operator)
  * [Walkthrough](#tumbling_walkthrough)
* [Hopping windows](#hopping)
  * [expire_hopping()](#expire_hopping-operator)
  * [group_by_agg_hopping()](#group_by_agg_hopping-operator)
  * [Walkthrough](#hopping_walkthrough)
* [Cumulative windows](#cumulative)
  * [expire_cumulative()](#expire_cumulative-operator)
  * [group_by_agg_cumulative()](#group_by_agg_cumulative-operator)
  * [Walkthrough](#cumulative_walkthrough)
* [Sliding windows](#sliding)
  * [expire_sliding()](#expire_sliding-operator)
  * [group_by_agg_sliding()](#group_by_agg_sliding-operator)
  * [Walkthrough](#sliding_walkthrough)
* [Session windows](#session)
  * [expire_session()](#expire_session-operator)
  * [group_by_agg_session()](#group_by_agg_session-operator)
  * [Walkthrough](#session_walkthrough.ipynb)
* [Triggers and custom windows](#custom)
  * [Session/Threshold windows](#threshold)
  * [Walkthrough](#threshold_walkthrough)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [ ]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator, OrderGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()
order_generator = OrderGenerator()

click_source_str = "clicks"
customer_source_str = "customers"
order_source_str = "orders"
sink_str = "sink"

#

def run(built_tn):
    sink_m_list = []
    for i in range(100):
        # 1. Generate new data.
        click_m_list = click_generator.generate(100)
        customer_m_list = customer_generator.generate(100)

        # 2. Push the new data to the topology + incrementally process the new data + get the resulting changes.
        sink_str_m_list_dict = built_tn.process({click_source_str: click_m_list, customer_source_str: customer_m_list})
        m_list = sink_str_m_list_dict[sink_str]

        # 3. Print out the size of the pydbsp state.
        sys.stdout.write(f"\rStep: {i + 1}, Memory: {built_tn.get_state_size() / 1024}KB")

        # 4. Add the changes to the output list.
        sink_m_list += m_list

    print()
    print(len(sink_m_list))
    print(sink_m_list[-10:])

def process(built_tn, customer_id, price, ts, w=1):
    m = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    #
    sink_str_r_list_dict = built_tn.process({order_source_str: [(m, w)]})
    r_list = sink_str_r_list_dict[sink_str]
    #
    print("Triggers:")
    for r in r_list:
        print(r)


---
<a id="expiry"></a>
## Expiry

Expiry is the central concept in Kafi Streams for freeing memory of timed out data.

Thanks to pydbsp, Kafi Streams can implement expiry natively, without having to bolt on any kind of mechanism on top.

Essentially, expiry has to be defined only once for each source at the beginning of the Kafi Streams topology. All the stateful operators downstream do not need any special handling - they are automatically cleaned up by the expired records percolating through the topology, one by one.

We need an example. Let us recollect the example from the [Quickstart](../quickstart.ipynb) using the `TopologyNode` class (see [Architecture](../architecture.ipynb))


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .upsert()
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

sink_tn = (
    click_tn
    .join(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

tn = Tn.build(sink_tn)


And then, let us throw data at it and see how the global state size of the topology grows:

In [ ]:
tn.reset()
for _ in range(3):
    run(tn)

This is of course not sustainable. The memory usage grows unboundedly and Kafi Streams gets slower and slower (in this case, mostly caused by the join).

It's time to introduce the `expire()` operator.

<a id="expire-operator"></a>
### expire()

Expires individual records so that they can be purged from memory:
```python
def expire(self, ts_fun, expiry_fun, project_fun=lambda r_end_ts_tuple: r_end_ts_tuple[0], **kwargs):
    """Expire records once the global watermark passes their expiry timestamp.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        expiry_fun: ts -> expiry ts - get expiry function
        project_fun: (r, end_ts) -> r - projection function (default: lambda r_end_ts_tuple: r_end_ts_tuple[0] i.e., drop the window end timestamp)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Let's add this to our topology.


In [ ]:
click_source_str = "clicks"
customer_source_str = "customers"

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    ###
    # expire() operator - expire after 1000 click generator steps
    ###
    .expire(ts_fun=lambda r: r["ts"],
            expiry_fun=lambda ts: ts + click_generator.ts_step_int * 1000)
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .upsert()
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

sink_tn = (
    click_tn
    .join(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

tn = Tn.build(sink_tn)


Here, we use the `expiry()` operator to:
1. Select the `ts` field from each record,
2. and then set the expiry to the selected timestamp plus `1000` times the timestamp step size of the click generator.

When you look closer at the topology, you can observe multiple some of the properties of expiry in Kafi Streams:
* It is defined once at the top of the topology, before any stateful operator (`map()` and `filter()` are stateless).
* The stateful operators (`distinct()` and `join()`) below in the topology do not need to know anything about the expiry.

Let's first see the graphical representation where you can see that the `expire()` operator also, under the covers, adds two `map()` operators (one upstream to add expiry timestamps and one downstream to project them):
```mermaid
graph TD
91e7790b-f2e8-4b17-89a1-dc8674903454[distinct_op] --> 0b66caac-9b22-497b-94df-dfc1353960ba[join_op]
5bdef638-4131-446b-bb10-0739280036b0[map_op] --> 2b034ba5-c3af-4e3e-9465-d757c0a045f4[expire_op]
0b66caac-9b22-497b-94df-dfc1353960ba[join_op] --> 662daecc-7690-4b3f-80e3-1cbdceef9af6[sink_sink]
c2a7546a-66c1-4d6a-94b1-81a8c6e26225[map_op] --> d2c470b4-3bfe-4cbc-964f-d8723cedce29[distinct_op]
5c13aa55-7c0b-4e04-a82d-55bf3796d955[source_customers] --> 9180e616-017f-4c88-a826-c3a124819e01[map_op]
2b034ba5-c3af-4e3e-9465-d757c0a045f4[expire_op] --> c2a7546a-66c1-4d6a-94b1-81a8c6e26225[map_op]
f301ae6b-838e-445c-afc3-f6a52f9ee28b[map_op] --> 866a2320-1e78-4276-83ed-8a60a2e476f7[filter_op]
c2431c19-64ab-4bdb-93e3-43826c60f592[source_clicks] --> f301ae6b-838e-445c-afc3-f6a52f9ee28b[map_op]
9180e616-017f-4c88-a826-c3a124819e01[map_op] --> 91e7790b-f2e8-4b17-89a1-dc8674903454[distinct_op]
d2c470b4-3bfe-4cbc-964f-d8723cedce29[distinct_op] --> 0b66caac-9b22-497b-94df-dfc1353960ba[join_op]
866a2320-1e78-4276-83ed-8a60a2e476f7[filter_op] --> 5bdef638-4131-446b-bb10-0739280036b0[map_op]
```

Ok. Let's see this in action. Will be able to rein in the memory consumption?

In [ ]:
for _ in range(5):
    run(tn)

It works! Constant, flat memory usage! No processing slowdown!

Why? Because we used `expire()` to time out the transactional data (=the clicks). After a short while, the memory consumption of the master data (=the customers) becomes constant as well because it is limited (the generator only generates up to 100 customers in the example).

---
<a id="windows"></a>
## Time windows

In the previous section, we learnt how we can keep Kafi Streams' memory usage at check. It was only remotely related to time windows in the classical stream processing sense: under the covers, the `expire()` operator works akin to a "sliding window" in classical stream processing.

This section is about "real" stream processing time windows.

You'll see that we devised a novel formulation of them inside DBSP that allows us to build all the time window types from classical stream processing.

But it doesn't stop there - Kafi Streams is so flexible that you can easily build your own custom time windows.


<a id="expire_group_aggregate"></a>
### Time Windows = expire + group + aggregate

What is a time window really? You can think of time windows in stream processing as consisting of two ingredients:
* **Expiry**: Time windows have a start and an end. Events *expire* after the end of a time window so that they can be cleaned up.
* **Group By + Aggregate**: The actual "time window" is a set of events *grouped* by time and some other key (e.g. a customer ID) and *aggregated*.

Now this is very theoretical. Let's pick the simplest time window - the *tumbling window* and see how all this theory plays out in practice.

In [ ]:
def ts_fun(r):
    return r["ts"]

size_int = click_generator.ts_step_int * 1000

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    ###
    # expire_tumbling() operator - expire tumbling windows with window size size_int    
    ###
    .expire_tumbling(ts_fun=ts_fun,
                     size_int=size_int)
    .distinct()
)

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

joined_tn = (
    click_tn
    .join(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]})
)

sink_tn = (
    joined_tn
    ###
    # group_by_agg_tumbling() operator:
    #   * group by composite key (customer_id and name)
    #   * aggregate:
    #     * clicks: count of the click records
    #     * view_times: collection of the view times
    #     * total_view_time: total sum of the view times
    #   * projection:
    #     * customer_id and name (composite group key), clicks and view_times and total_view_time (aggregation)
    ###
    .group_by_agg_tumbling(
        ts_fun=ts_fun,
        size_int=size_int,
        key_fun=lambda r: {"customer_id": r["customer_id"], "name": r["name"]},
        agg_fun=lambda agg_r, r: {"clicks": agg_r["clicks"] + 1,
                                  "view_times": agg_r["view_times"] + [r["view_time"]],
                                  "total_view_time": agg_r["total_view_time"] + r["view_time"]},
        agg_initial_any={"clicks": 0, "view_times": [], "total_view_time": 0},
        project_fun=lambda key_any, agg_r: {"customer_id": key_any["customer_id"],
                                            "name": key_any["name"],
                                            "clicks": agg_r["clicks"],
                                            "view_times": agg_r["view_times"],
                                            "total_view_time": agg_r["total_view_time"]})
    .sink(sink_str)
)

tn = Tn.build(sink_tn)


What do we do here?

* **expire**: At the top of the topology, we specify the expiry of the incoming clicks using the `expire_tumbling()` operator and set the tumbling window size to `tumbling_size_int = click_generator.ts_step_int * 1000`.
* **group + aggregate**: After the join of clicks and customers, we create the tumbling window using the `group_by_agg_tumbling()` operator: We group by `customer_id` and `name`, and aggregate the clicks for that customer in that time window:
  * `clicks` the number of clicks of the customer
  * `view_times` the list of view times of the customer
  * `total_view_time` the total view time of the customer

Let's see this graphically:
```mermaid
graph TD
5a41a8b2-6516-453d-b5bc-c7d3d150a732[distinct_op] --> 37271487-3d33-426b-b2cf-786f2635dec3[join_op]
cd4a645f-4e8f-46ce-bec4-86d42b5fae9f[join_pred_op] --> bea060b3-c8e2-4490-b6a0-7f20131fdf63[_filter_op]
c1f230f0-a5be-44cd-ac8a-397988ba1b12[map_op] --> 4a3c98b6-4519-4223-975f-51ec4b40d744[distinct_op]
6e5943e9-a1ba-4ea5-b850-9d2f88995626[source_customers] --> 2c3fe350-86f3-4e5c-a0dc-2372e7773daf[map_op]
1e47b915-1400-4a27-a66a-82b13edabe02[expire_op] --> c1f230f0-a5be-44cd-ac8a-397988ba1b12[map_op]
2c3fe350-86f3-4e5c-a0dc-2372e7773daf[map_op] --> 5a41a8b2-6516-453d-b5bc-c7d3d150a732[distinct_op]
e66f3220-1db1-468b-9c66-3ca579a922a0[map_op] --> 086e0a0f-167b-4c0c-80a7-ec2e551e842a[filter_op]
086e0a0f-167b-4c0c-80a7-ec2e551e842a[filter_op] --> 5f230b63-a718-4095-84d5-7520956a3a87[map_op]
bbd161a2-f49c-4880-a433-2443154dac26[source_clicks] --> e66f3220-1db1-468b-9c66-3ca579a922a0[map_op]
5f230b63-a718-4095-84d5-7520956a3a87[map_op] --> 1e47b915-1400-4a27-a66a-82b13edabe02[expire_op]
c26e90a5-fd3f-42ac-9212-c5df783301a5[flatmap_op] --> 9ae3a291-f77c-4790-b28e-2c6b0d16289d[group_by_agg_op]
4a3c98b6-4519-4223-975f-51ec4b40d744[distinct_op] --> 37271487-3d33-426b-b2cf-786f2635dec3[join_op]
bea060b3-c8e2-4490-b6a0-7f20131fdf63[_filter_op] --> a788da33-66e2-4b43-a8b5-9d89bcc9d15c[sink_sink]
9ae3a291-f77c-4790-b28e-2c6b0d16289d[group_by_agg_op] --> cd4a645f-4e8f-46ce-bec4-86d42b5fae9f[join_pred_op]
e69bb8ec-0d1a-4a1a-bc6f-359eb0e80920[max_op] --> cd4a645f-4e8f-46ce-bec4-86d42b5fae9f[join_pred_op]
37271487-3d33-426b-b2cf-786f2635dec3[join_op] --> e69bb8ec-0d1a-4a1a-bc6f-359eb0e80920[max_op]
37271487-3d33-426b-b2cf-786f2635dec3[join_op] --> c26e90a5-fd3f-42ac-9212-c5df783301a5[flatmap_op]
```

And let's run this.

In [ ]:
run(tn)

This was your first time window in Kafi Streams in action!

In the following sections, we double down on the individual built-in window types of Kafi Streams and explain their API in detail.

<a id="watermarks"></a>
### Watermarks

In this version, Kafi Streams it does *not* use implicit *watermarks* for tracking the progress of time throughout the topology. The approximation of a watermark in Kafi Streams is the maximum timestamp of the messages that it has seen so far - in the following we refer to this as the *latest timestamp*.

This is a simplification and also clearly a trade-off. It makes the architecture of Kafi Streams simpler, but without implicit watermarking, Kafi Streams can drop time windows in case the gaps between the source events are bigger than the respective window size + allowed lateness.

There are workarounds for this, but, as not using implicit watermarks is a trade-off, none of them comes without downsides. Here are two:
* Increase the allowed lateness, with the disadvantage of higher memory consumption.
* Augment the source events with events that serve as "explicit watermarks", with the disadvantege of handing over this part of the complexity to the producer (e.g. dummy events that you put in regular intervals less than than window size + allowed lateness).


---
<a id="tumbling"></a>
## Tumbling Windows

In Kafi Streams, the two operators required to set up a tumbling window are `expire_tumbling` and `group_by_agg_tumbling`.

<a id="expire_tumbling-operator"></a>
### expire_tumbling()

Syntactic sugar for `expire()` for record expiry in the context of tumbling windows:
```python
def expire_tumbling(self, ts_fun, size_int, allowed_lateness_int=0, **kwargs):
    """Expire records once past their tumbling window.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        allowed_lateness_int: allowed lateness
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

Using this operator, the expiry time for a record with timestamp `ts` is: `(ts // size_int) * size_int + size_int + size_int + allowed_lateness_int`. Note that a second `size_int` needs to be added to avoid the record being discarded too early. We call this additional addition *window buffer time*.


<a id="group_by_agg_tumbling-operator"></a>
### group_by_agg_tumbling()

Augmented `group_by_agg()` operator for creating tumbling time windows by grouping by + aggregating. Implicitly also groups by the time windows and triggers the emission of aggregated time windows:
```python
def group_by_agg_tumbling(self, ts_fun, size_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1], trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs):
    """Tumbling window aggregation, emitted once each window closes or using a custom trigger function.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        key_fun: r -> key_any - grouping key function
        agg_fun: (agg_r, r) -> agg_r - aggregate function
        agg_initial_any: initial aggregate
        project_fun: key_any, agg_r -> r - projection function
        trigger_fun: ((r, end ts), latest ts) -> bool, predicate to trigger the emission of a window (default: lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1])
        project_fun: (r, end_ts) -> projection function (default: lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]})
        positive_only_bool: if True, suppress retractions (w <= 0) from the output (default: True)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

This looks scary at first. But for most use cases, only these parameters are obligatory:
* `ts_fun` - same as in `expire_tumbling()`
* `size_int` - same as in `expire_tumbling()`
* `key_fun` - as in `group_by_agg()`
* `agg_fun` - as in `group_by_agg()`
* `agg_initial_any` - as in `group_by_agg()`
* `project_fun` - as in `group_by_agg()`

The `trigger_` parameters should only be required for advanced use cases. They control the emission of time windows based on a triggering mechanism. Their defaults should suffice for most uses cases. We'll show a use case for them when we discuss custom time windows at the end of this notebook.

Note that contrary to the basic `group_by_agg()` operator, there is no `value_fun`. This is because the `value_fun` in `group_by_agg_tumbling` is always the identity function since we assume that in 99% of the use cases, you would want to create time windows around entire records.


<a id="tumbling_walkthrough"></a>
### Walkthrough

Next, you can go through an exhaustive illustration of how Kafi Streams' tumbling windows work by walking through some example data - one by one, in baby steps, graphically: [tumbling window walkthrough](windows/tumbling.ipynb)


---
<a id="hopping"></a>
## Hopping Windows

In Kafi Streams, the two operators required to set up a hopping window are `expire_hopping` and `group_by_agg_hoppling`.

<a id="expire_hopping-operator"></a>
### expire_hopping()

Syntactic sugar for `expire()` for record expiry in the context of hopping windows:
```python
def expire_hopping(self, ts_fun, size_int, hop_int, allowed_lateness_int=0, **kwargs):
    """Expire records once past their latest hopping window.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        hop_int: hop between windows, hop_int <= size_int
        allowed_lateness_int: allowed lateness
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

`expire_hopping` has the same signature as `expire_tumbling` except for the additional `hop_int` parameter to specify the hop size.

Using this operator, the expiry time for a record with timestamp `ts` is: `(ts // size_int) * size_int + size_int + size_int + allowed_lateness_int`. Note that a second `size_int` again needs to be added to avoid the record being discarded too early (window buffer time).


<a id="group_by_agg_hopping-operator"></a>
### group_by_agg_hopping()

Augmented `group_by_agg()` operator for creating hopping time windows by grouping by + aggregating. Implicitly also groups by the time windows and triggers the emission of aggregated time windows:
```python
def group_by_agg_hopping(self, ts_fun, size_int, hop_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1], trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs):
    """Hopping window aggregation, emitted once each window closes or using a custom trigger function.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        hop_int: hop size
        key_fun: r -> key_any - grouping key function
        agg_fun: (agg_r, r) -> agg_r - aggregate function
        agg_initial_any: initial aggregate
        project_fun: key_any, agg_r -> r - projection function
        trigger_fun: ((r, end ts), latest ts) -> bool, predicate to trigger the emission of a window (default: lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1])
        project_fun: (r, end_ts) -> projection function (default: lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]})
        positive_only_bool: if True, suppress retractions (w <= 0) from the output (default: True)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

`group_by_agg_hopping` has the same signature as `group_by_tumbling` except for the additional `hop_int` parameter to specify the hop size.


<a id="hopping_walkthrough"></a>
### Walkthrough

Next, you can go through an exhaustive illustration of how Kafi Streams' hopping windows work by walking through some example data - one by one, in baby steps, graphically: [hopping window walkthrough](windows/hopping.ipynb)


---
<a id="cumulative"></a>
## Cumulative Windows

In Kafi Streams, the two operators required to set up a cumulative window are `expire_cumulative` and `group_by_agg_cumulative`.

<a id="expire_cumulative-operator"></a>
### expire_cumulative()

Syntactic sugar for `expire()` for record expiry in the context of cumulative windows:
```python
def expire_cumulative(self, ts_fun, size_int, step_int, allowed_lateness_int=0, **kwargs):
    """Expire records once past their cumulative window.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        step_int: step size between windows
        allowed_lateness_int: allowed lateness
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

`expire_cumulative` has the same signature as `expire_tumbling` except for the additional `step_int` parameter to specify the step size.

Using this operator, the expiry time for a record with timestamp `ts` is: `(ts // size_int) * size_int + size_int + size_int + allowed_lateness_int`. Note that a second `size_int` again needs to be added to avoid the record being discarded too early (window buffer time).


<a id="group_by_agg_cumulative-operator"></a>
### group_by_agg_cumulative()

Augmented `group_by_agg()` operator for creating cumulative time windows by grouping by + aggregating. Implicitly also groups by the time windows and triggers the emission of aggregated time windows:
```python
def group_by_agg_cumulative(self, ts_fun, size_int, step_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1], trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs):
    """Cumulative window aggregation, emitted once each window closes or using a custom trigger function.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        step_int: step size
        key_fun: r -> key_any - grouping key function
        agg_fun: (agg_r, r) -> agg_r - aggregate function
        agg_initial_any: initial aggregate
        project_fun: key_any, agg_r -> r - projection function
        trigger_fun: ((r, end ts), latest ts) -> bool, predicate to trigger the emission of a window (default: lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1])
        project_fun: (r, end_ts) -> projection function (default: lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]})
        positive_only_bool: if True, suppress retractions (w <= 0) from the output (default: True)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

`group_by_agg_cumulative` has the same signature as `group_by_tumbling` except for the additional `step_int` parameter to specify the step size.


<a id="cumulalive_walkthrough"></a>
### Walkthrough

Next, you can go through an exhaustive illustration of how Kafi Streams' cumulative windows work by walking through some example data - one by one, in baby steps, graphically: [cumulative window walkthrough](windows/cumulative.ipynb)


---
<a id="sliding"></a>
## Sliding Windows

In Kafi Streams, the two operators required to set up a sliding window are `expire_sliding` and `group_by_agg_sliding`.

<a id="expire_sliding-operator"></a>
### expire_sliding()

Syntactic sugar for `expire()` for record expiry in the context of sliding windows:
```python
def expire_sliding(self, ts_fun, size_int, allowed_lateness_int, **kwargs):
    """Expire records once past their own sliding window.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        allowed_lateness_int: allowed lateness
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

`expire_sliding` has the same signature as `expire_tumbling`.

Using this operator, the expiry time for a record with timestamp `ts` is: `ts + size_int + allowed_lateness_int`. Contrary to tumbling, hopping and cumulative windows, but similar to session windows, no extra window buffer time is required.


<a id="group_by_agg_sliding-operator"></a>
### group_by_agg_sliding()

Augmented `group_by_agg()` operator for creating sliding time windows by grouping by + aggregating. Implicitly also groups by the time windows and triggers the emission of aggregated time windows:
```python
def group_by_agg_sliding(self, ts_fun, size_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs):
    """Sliding window aggregation; emitted once each window closes or using a custom trigger function.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        key_fun: r -> key_any - grouping key function
        agg_fun: (agg_r, r) -> agg_r - aggregate function
        agg_initial_any: initial aggregate
        project_fun: key_any, agg_r -> r - projection function
        trigger_fun: ((r, end ts), latest ts) -> bool, predicate to trigger the emission of a window (default: lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1])
        project_fun: (r, end_ts) -> projection function (default: lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]})
        positive_only_bool: if True, suppress retractions (w <= 0) from the output (default: True)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

`group_by_agg_sliding` has the same signature as `group_by_tumbling`.


<a id="sliding_walkthrough"></a>
### Walkthrough

Next, you can go through an exhaustive illustration of how Kafi Streams' sliding windows work by walking through some example data - one by one, in baby steps, graphically: [sliding window walkthrough](windows/sliding.ipynb)


---
<a id="session"></a>
## Session Windows

In Kafi Streams, the two operators required to set up a session window are `expire_session` and `group_by_agg_session`.

<a id="expire_session-operator"></a>
### expire_session()

Syntactic sugar for `expire()` for record expiry in the context of session windows:
```python
def expire_session(self, ts_fun, max_session_int, allowed_lateness_int=0, **kwargs):
    """Expire records once past their session boundary.
        
    Args:
        ts_fun: r -> ts - get timestamp function
        max_session_int: grid size used internally while bucketing session windows
        allowed_lateness_int: allowed lateness
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

`expire_session` has the same signature as `expire_sliding` except that the `size_int` parameter is called `max_session_int` to signify that it doesn't specify the window size but the length of a maximum session.

Using this operator, the expiry time for a record with timestamp `ts` is: `(ts // max_session_int) * max_session_int + max_session_int + allowed_lateness_int`. Contrary to tumbling, hopping and cumulative windows, but similar to sliding windows, no extra window buffer time is required.


<a id="group_by_agg_session-operator"></a>
### group_by_agg_session()

Augmented `group_by_agg()` operator for creating session time windows by grouping by + aggregating. Implicitly also groups by the time windows and triggers the emission of aggregated time windows:
```python
def group_by_agg_session(self, ts_fun, gap_int, key_fun, agg_fun, agg_initial_any, project_fun, trigger_fun=lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1], trigger_project_fun=lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]}, trigger_positive_only_bool=True, **kwargs):
    """Session window aggregation, emitted once each session closes or using a custom trigger function.
    
    Args:
        ts_fun: r -> ts - get timestamp function
        size_int: window size
        gap_int: gap size
        key_fun: r -> key_any - grouping key function
        agg_fun: (agg_r, r) -> agg_r - aggregate function
        agg_initial_any: initial aggregate
        project_fun: key_any, agg_r -> r - projection function
        trigger_fun: ((r, end ts), latest ts) -> bool, predicate to trigger the emission of a window (default: lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1])
        project_fun: (r, end_ts) -> projection function (default: lambda r_end_ts_tuple: {**r_end_ts_tuple[0], "window_end": r_end_ts_tuple[1]})
        positive_only_bool: if True, suppress retractions (w <= 0) from the output (default: True)
        **kwargs: passed through to the underlying node(s)
    Returns:
        tn: the newly created topology node of the operator"""
```

`group_by_agg_session` has the same signature as `group_by_tumbling` except that the `size_int` parameter is called `gap_int` to signify that it doesn't specify the window size but the length of the allowed gap between two session windows:



<a id="session_walkthrough"></a>
### Walkthrough

Next, you can go through an exhaustive illustration of how Kafi Streams' session windows work by walking through some example data - one by one, in baby steps, graphically: [session windows walkthrough](windows/sliding.ipynb)


---
<a id="custom"></a>
## Triggers and custom windows

With Kafi Streams' you can easily build new kinds of time windows based on the existing ones.

The main lever that Kafi Streams lays into your hands is the trigger function controlling the emission of time window outputs.

However, you could in principle also add completely different kinds of time windows to Kafi Streams' `TopologyNode` class if you'd like to.

This section contains a typical example: A custom *session/threshold* window.

<a id="threshold"></a>
### Session/threshold windows

A session/threshold window is simply a session window that fires not only when a session window is over but can also fire earlier if some kind of threshold has been exceeded.

The key for setting up this custom kind of window is the trigger function - essentially by `or`ing it with the threshold condition.

A throrough example is provided in the walkthrough below.

<a id="threshold_walkthrough"></a>
### Walkthrough

[session windows walkthrough](windows/threshold.ipynb)
